# Bioprocessing and Bioenergy in NeqSim

This notebook demonstrates NeqSim's bioprocessing capabilities:

1. **Anaerobic Digestion** — biogas from food waste
2. **Biogas Upgrading** — purification to grid-quality biomethane
3. **Fermentation Kinetics** — ethanol production with Monod model
4. **Sustainability Metrics** — CO₂eq tracking and carbon intensity
5. **Biogas-to-Grid Module** — complete pre-built process chain
6. **Waste-to-Energy CHP** — combined heat and power from waste

In [ ]:
# Setup: Import NeqSim
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except ImportError:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import jpype
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Import NeqSim classes
from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

# Bioenergy classes
AnaerobicDigester = jneqsim.process.equipment.reactor.AnaerobicDigester
BiomassCharacterization = jneqsim.thermo.characterization.BiomassCharacterization
BiogasUpgrader = jneqsim.process.equipment.splitter.BiogasUpgrader
FermentationReactor = jneqsim.process.equipment.reactor.FermentationReactor

# Sustainability
SustainabilityMetrics = jpype.JClass("neqsim.process.util.fielddevelopment.SustainabilityMetrics")

# Pre-built modules
BiogasToGridModule = jpype.JClass("neqsim.process.processmodel.biorefinery.BiogasToGridModule")
WasteToEnergyCHPModule = jpype.JClass("neqsim.process.processmodel.biorefinery.WasteToEnergyCHPModule")

print("All classes loaded successfully")

## 1. Anaerobic Digestion

Model biogas production from food waste at mesophilic temperature (37°C).

In [ ]:
# Create a simple feed stream (water-based proxy for organic waste)
feed_fluid = SystemSrkEos(273.15 + 35.0, 1.01325)
feed_fluid.addComponent("water", 80.0)
feed_fluid.addComponent("methane", 0.01)  # trace dissolved methane
feed_fluid.setMixingRule("classic")

feed = Stream("Waste Feed", feed_fluid)
feed.setFlowRate(10000.0, "kg/hr")  # 10 tonnes/hr
feed.run()

# Set up anaerobic digester
digester = AnaerobicDigester("AD-1", feed)
digester.setSubstrateType(AnaerobicDigester.SubstrateType.FOOD_WASTE)
digester.setDigesterTemperature(37.0, "C")
digester.setFeedRate(10000.0, 0.25)  # 10 t/hr, 25% total solids
digester.setVesselVolume(6000.0)  # m3

process = ProcessSystem()
process.add(digester)
process.run()

biogas = digester.getBiogasOutStream()
digestate = digester.getDigestateOutStream()

print("=== Anaerobic Digestion Results ===")
print(f"Biogas flow rate: {biogas.getFlowRate('kg/hr'):.1f} kg/hr")
print(f"Digestate flow rate: {digestate.getFlowRate('kg/hr'):.1f} kg/hr")
print(f"Methane production: {digester.getMethaneProductionNm3PerDay():.0f} Nm3/day")
print(f"Biogas flow rate: {digester.getBiogasFlowRateNm3PerDay():.0f} Nm3/day")

## 2. Biogas Upgrading

Compare different upgrading technologies for biomethane quality.

In [ ]:
technologies = [
    ("Water Scrubbing", BiogasUpgrader.UpgradingTechnology.WATER_SCRUBBING),
    ("Amine Scrubbing", BiogasUpgrader.UpgradingTechnology.AMINE_SCRUBBING),
    ("PSA", BiogasUpgrader.UpgradingTechnology.PSA),
    ("Membrane", BiogasUpgrader.UpgradingTechnology.MEMBRANE),
]

tech_names = []
ch4_recoveries = []
co2_removals = []

for name, tech in technologies:
    upgrader = BiogasUpgrader(f"Upgrader_{name}", biogas)
    upgrader.setTechnology(tech)
    upgrader.run()

    biomethane = upgrader.getBiomethaneOutStream()
    offgas = upgrader.getOffgasOutStream()

    bm_flow = biomethane.getFlowRate("kg/hr")
    og_flow = offgas.getFlowRate("kg/hr")
    recovery = float(tech.getCh4Recovery()) * 100
    removal = float(tech.getCo2Removal()) * 100

    tech_names.append(name)
    ch4_recoveries.append(recovery)
    co2_removals.append(removal)

    print(f"{name}: Biomethane={bm_flow:.1f} kg/hr, Offgas={og_flow:.1f} kg/hr, "
          f"CH4 recovery={recovery:.0f}%, CO2 removal={removal:.0f}%")

In [ ]:
# Plot: Compare upgrading technologies
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tech_names))
width = 0.35
bars1 = ax.bar(x - width/2, ch4_recoveries, width, label='CH4 Recovery (%)', color='#2196F3')
bars2 = ax.bar(x + width/2, co2_removals, width, label='CO2 Removal (%)', color='#FF9800')
ax.set_ylabel('Percentage (%)')
ax.set_title('Biogas Upgrading Technology Comparison')
ax.set_xticks(x)
ax.set_xticklabels(tech_names)
ax.legend()
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Fermentation Kinetics

Model ethanol fermentation comparing Monod and substrate-inhibited kinetics.

In [ ]:
# Create a sugar feed stream
sugar_fluid = SystemSrkEos(273.15 + 30.0, 1.01325)
sugar_fluid.addComponent("water", 90.0)
sugar_fluid.addComponent("n-hexane", 10.0)  # proxy for glucose
sugar_fluid.setMixingRule("classic")

sugar_feed = Stream("Sugar Feed", sugar_fluid)
sugar_feed.setFlowRate(1000.0, "kg/hr")
sugar_feed.run()

# Monod kinetics - continuous fermentation
reactor_monod = FermentationReactor("FR-Monod", sugar_feed)
reactor_monod.setKineticModel(FermentationReactor.KineticModel.MONOD)
reactor_monod.setOperationMode(FermentationReactor.OperationMode.CONTINUOUS)
reactor_monod.setSubstrateConcentration(100.0)  # g/L
reactor_monod.setBiomassConcentration(1.0)       # g/L
reactor_monod.setMaxSpecificGrowthRate(0.30)     # 1/hr
reactor_monod.setMonodConstant(1.0)              # g/L
reactor_monod.setYieldBiomass(0.10)
reactor_monod.setYieldProduct(0.45)
reactor_monod.setResidenceTime(10.0, "hr")
reactor_monod.run()

results_monod = dict(reactor_monod.getResults())
print("=== Monod Continuous Fermentation ===")
print(f"Substrate conversion: {float(results_monod['substrateConversion'])*100:.1f}%")
print(f"Product concentration: {float(results_monod['finalProductConc_g_per_L']):.1f} g/L")
print(f"Biomass concentration: {float(results_monod['finalBiomassConc_g_per_L']):.1f} g/L")
print(f"Productivity: {float(results_monod['productivity_g_per_L_hr']):.2f} g/L/hr")

In [ ]:
# Sensitivity: effect of residence time on product concentration
residence_times = [4, 6, 8, 10, 12, 15, 20, 25, 30]
product_concs = []
conversions = []
productivities = []

for tau in residence_times:
    reactor = FermentationReactor(f"FR-tau{tau}", sugar_feed)
    reactor.setKineticModel(FermentationReactor.KineticModel.MONOD)
    reactor.setOperationMode(FermentationReactor.OperationMode.CONTINUOUS)
    reactor.setSubstrateConcentration(100.0)
    reactor.setBiomassConcentration(1.0)
    reactor.setMaxSpecificGrowthRate(0.30)
    reactor.setMonodConstant(1.0)
    reactor.setYieldBiomass(0.10)
    reactor.setYieldProduct(0.45)
    reactor.setResidenceTime(float(tau), "hr")
    reactor.run()

    res = dict(reactor.getResults())
    product_concs.append(float(res['finalProductConc_g_per_L']))
    conversions.append(float(res['substrateConversion']) * 100)
    productivities.append(float(res['productivity_g_per_L_hr']))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(residence_times, product_concs, 'bo-', label='Product Conc.')
ax1.set_xlabel('Residence Time (hr)')
ax1.set_ylabel('Product Concentration (g/L)', color='b')
ax1_r = ax1.twinx()
ax1_r.plot(residence_times, conversions, 'rs--', label='Conversion')
ax1_r.set_ylabel('Substrate Conversion (%)', color='r')
ax1.set_title('Product Concentration & Conversion vs Residence Time')
ax1.grid(alpha=0.3)

ax2.plot(residence_times, productivities, 'go-')
ax2.set_xlabel('Residence Time (hr)')
ax2.set_ylabel('Volumetric Productivity (g/L/hr)')
ax2.set_title('Productivity vs Residence Time')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sustainability Metrics

Calculate CO₂-equivalent emissions, carbon intensity, and net carbon balance
for a biogas CHP plant.

In [ ]:
metrics = SustainabilityMetrics()
metrics.setBiogasProductionNm3PerYear(3_000_000.0)  # 3 million Nm3/yr
metrics.setMethaneContentFraction(0.60)
metrics.setMethaneSlipPercent(1.5)
metrics.setElectricityProductionMWhPerYear(8000.0)
metrics.setHeatProductionMWhPerYear(10000.0)
metrics.setParasiticElectricityMWhPerYear(800.0)
metrics.setFossilReferenceEmissionFactor(0.450)  # kgCO2/kWh (= tCO2/MWh)
metrics.setFossilHeatEmissionFactor(0.250)
metrics.calculate()

print("=== Sustainability Analysis ===")
print(f"Total emissions: {metrics.getTotalEmissionsTCO2eqPerYear():.0f} tCO2eq/yr")
print(f"Carbon intensity: {metrics.getCarbonIntensityKgCO2PerMWh():.1f} kgCO2eq/MWh")
print(f"Fossil fuel displacement: {metrics.getFossilFuelDisplacementTCO2PerYear():.0f} tCO2/yr")
print(f"Net carbon balance: {metrics.getNetCarbonBalanceTCO2PerYear():.0f} tCO2/yr")
print(f"EROI: {metrics.getEnergyReturnOnInvestment():.1f}")
print(f"Renewable energy fraction: {metrics.getRenewableEnergyFraction()*100:.0f}%")

In [ ]:
# Compare plants with different methane slip rates
slip_rates = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
emissions_list = []
net_carbon_list = []
carbon_intensity_list = []

for slip in slip_rates:
    m = SustainabilityMetrics()
    m.setBiogasProductionNm3PerYear(3_000_000.0)
    m.setMethaneContentFraction(0.60)
    m.setMethaneSlipPercent(float(slip))
    m.setElectricityProductionMWhPerYear(8000.0)
    m.setHeatProductionMWhPerYear(10000.0)
    m.setParasiticElectricityMWhPerYear(800.0)
    m.setFossilReferenceEmissionFactor(0.450)
    m.setFossilHeatEmissionFactor(0.250)
    m.calculate()

    emissions_list.append(float(m.getTotalEmissionsTCO2eqPerYear()))
    net_carbon_list.append(float(m.getNetCarbonBalanceTCO2PerYear()))
    carbon_intensity_list.append(float(m.getCarbonIntensityKgCO2PerMWh()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(range(len(slip_rates)), emissions_list, color='#F44336', alpha=0.8)
ax1.set_xticks(range(len(slip_rates)))
ax1.set_xticklabels([f'{s}%' for s in slip_rates])
ax1.set_xlabel('Methane Slip Rate')
ax1.set_ylabel('Total Emissions (tCO2eq/yr)')
ax1.set_title('Emissions vs Methane Slip')
ax1.grid(axis='y', alpha=0.3)

colors = ['green' if nc < 0 else 'red' for nc in net_carbon_list]
ax2.bar(range(len(slip_rates)), net_carbon_list, color=colors, alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(len(slip_rates)))
ax2.set_xticklabels([f'{s}%' for s in slip_rates])
ax2.set_xlabel('Methane Slip Rate')
ax2.set_ylabel('Net Carbon Balance (tCO2/yr)')
ax2.set_title('Net Carbon Balance vs Methane Slip')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Biogas-to-Grid Module

Pre-built module: digestion → upgrading → compression → grid injection.

In [ ]:
# Create waste feed
grid_feed_fluid = SystemSrkEos(273.15 + 35.0, 1.01325)
grid_feed_fluid.addComponent("water", 75.0)
grid_feed_fluid.addComponent("methane", 0.01)
grid_feed_fluid.setMixingRule("classic")

grid_feed = Stream("Grid Feed", grid_feed_fluid)
grid_feed.setFlowRate(5000.0, "kg/hr")
grid_feed.run()

# Run biogas-to-grid module with membrane upgrading
btg = BiogasToGridModule("BTG-Plant")
btg.setFeedStream(grid_feed)
btg.setSubstrateType(AnaerobicDigester.SubstrateType.FOOD_WASTE)
btg.setUpgradingTechnology(BiogasUpgrader.UpgradingTechnology.MEMBRANE)
btg.setGridPressureBara(40.0)
btg.setGridTemperatureC(25.0)
btg.setDigesterTemperatureC(37.0)
btg.run()

results_btg = dict(btg.getResults())
print("=== Biogas-to-Grid Results ===")
for key, val in results_btg.items():
    print(f"  {key}: {val}")

## 6. Waste-to-Energy CHP

Pre-built module: anaerobic digestion + gas engine CHP.

In [ ]:
# Create waste feed
chp_feed_fluid = SystemSrkEos(273.15 + 35.0, 1.01325)
chp_feed_fluid.addComponent("water", 80.0)
chp_feed_fluid.addComponent("methane", 0.01)
chp_feed_fluid.setMixingRule("classic")

chp_feed = Stream("CHP Feed", chp_feed_fluid)
chp_feed.setFlowRate(8000.0, "kg/hr")
chp_feed.run()

# Run CHP module
chp = WasteToEnergyCHPModule("CHP-Plant")
chp.setFeedStream(chp_feed)
chp.setSubstrateType(AnaerobicDigester.SubstrateType.FOOD_WASTE)
chp.setElectricalEfficiency(0.40)
chp.setThermalEfficiency(0.45)
chp.setOperatingHoursPerYear(8000.0)
chp.run()

print("=== Waste-to-Energy CHP Results ===")
print(f"Electrical power: {chp.getElectricalPowerKW():.1f} kW")
print(f"Heat output: {chp.getHeatOutputKW():.1f} kW")
print(f"Total CHP efficiency: {chp.getTotalCHPefficiency()*100:.0f}%")
print(f"CO2 emissions: {chp.getCO2EmissionsKgPerHr():.1f} kg/hr")
print(f"Annual electricity: {chp.getAnnualElectricityMWh():.0f} MWh/yr")
print(f"Annual heat: {chp.getAnnualHeatMWh():.0f} MWh/yr")

In [ ]:
# Compare substrates for CHP output
substrates = [
    ("Food Waste", AnaerobicDigester.SubstrateType.FOOD_WASTE),
    ("Sewage Sludge", AnaerobicDigester.SubstrateType.SEWAGE_SLUDGE),
    ("Manure", AnaerobicDigester.SubstrateType.MANURE),
    ("Crop Residue", AnaerobicDigester.SubstrateType.CROP_RESIDUE),
]

sub_names = []
elec_powers = []
heat_powers = []

for name, sub_type in substrates:
    m = WasteToEnergyCHPModule(f"CHP-{name}")
    m.setFeedStream(chp_feed)
    m.setSubstrateType(sub_type)
    m.setElectricalEfficiency(0.38)
    m.setThermalEfficiency(0.45)
    m.run()

    sub_names.append(name)
    elec_powers.append(float(m.getElectricalPowerKW()))
    heat_powers.append(float(m.getHeatOutputKW()))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(sub_names))
width = 0.35
ax.bar(x - width/2, elec_powers, width, label='Electrical (kW)', color='#4CAF50')
ax.bar(x + width/2, heat_powers, width, label='Thermal (kW)', color='#FF5722')
ax.set_ylabel('Power Output (kW)')
ax.set_title('CHP Output by Substrate Type (8 t/hr feed)')
ax.set_xticks(x)
ax.set_xticklabels(sub_names, rotation=15)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

NeqSim's bioprocessing toolkit provides:

| Capability | Key Class |
|-----------|----------|
| Biomass characterization | `BiomassCharacterization` |
| Anaerobic digestion | `AnaerobicDigester` |
| Thermochemical gasification | `BiomassGasifier` |
| Pyrolysis | `PyrolysisReactor` |
| Biogas upgrading | `BiogasUpgrader` |
| Fermentation kinetics | `FermentationReactor` |
| Sustainability/LCA | `SustainabilityMetrics` |
| Cost estimation | `BiorefineryCostEstimator` |
| Biogas-to-grid chain | `BiogasToGridModule` |
| Gasification + FT | `GasificationSynthesisModule` |
| Waste-to-energy CHP | `WasteToEnergyCHPModule` |

For more details, see the [bioprocessing documentation](../../docs/process/bioprocessing.md).